# Creación de una aplicación web con Flask y SQLite3

## Aplicación web de gestión de tareas

### Objetivo

Desarrollar y documentar una aplicación web completa utilizando el microframework **Flask** y una
base de datos **SQLite3** gestionada mediante Flask-SQLAlchemy, demostrando la integración entre
un servidor web Python, un ORM relacional y plantillas HTML dinámicas.

### Instrucciones

Investigar y construir una aplicación web de lista de tareas (*todo list*) que incluya:

- Configuración del entorno Flask y conexión a una base de datos SQLite3.
- Definición de un modelo ORM con Flask-SQLAlchemy.
- Implementación de rutas para las operaciones CRUD (Crear, Leer, Actualizar, Eliminar).
- Uso de plantillas Jinja2 para renderizar vistas dinámicas.
- Despliegue de la aplicación en un servicio en la nube (Render).

Presenta los resultados en un archivo que incluya:

- Código fuente comentado.
- Una breve descripción (de 2-3 párrafos) explicando cómo se aplicaron los conceptos.

### Criterios de Evaluación

- Correcta configuración y uso de Flask como servidor web.
- Diseño adecuado del modelo de datos con SQLAlchemy.
- Implementación funcional de las operaciones CRUD.
- Uso correcto de plantillas Jinja2 y redirecciones HTTP.
- Documentación clara y concisa.

---

**Repositorio del proyecto:** https://github.com/RKCbas/todo-flask
**Aplicación desplegada:** https://todo-flask-zkex.onrender.com
**Repositorio de prácticas:** https://github.com/RKCbas/Maestria-en-Inteligencia-Artificial---Practicas

## Entorno y Dependencias

Este notebook documenta una aplicación Flask desplegada en producción. Las dependencias
principales son **Flask 3.1.2**, **Flask-SQLAlchemy 3.1.1** y **gunicorn 23.0.0**
(exclusivo de producción). La celda siguiente verifica que las bibliotecas estén disponibles
en el entorno local.

```
Flask==3.1.2
Flask-SQLAlchemy==3.1.1
gunicorn==23.0.0
```

Para instalar las dependencias:

```bash
pip install Flask Flask-SQLAlchemy gunicorn
```

In [3]:
import sys
import importlib

DEPENDENCIAS = {
    'flask': 'Flask',
    'flask_sqlalchemy': 'Flask-SQLAlchemy',
    'sqlalchemy': 'SQLAlchemy',
}

print(f'Python {sys.version}')
print(f'Plataforma: {sys.platform}')
print('-' * 55)
print('Paquete                Version         Estado')
print('-' * 55)
for modulo, nombre in DEPENDENCIAS.items():
    try:
        mod = importlib.import_module(modulo)
        version = getattr(mod, '__version__', 'instalado')
        print(f'  {nombre:<20} {version:<16} disponible')
    except ImportError:
        print(f'  {nombre:<20} {chr(8212):<16} NO instalado')
print('-' * 55)

Python 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Plataforma: win32
-------------------------------------------------------
Paquete                Version         Estado
-------------------------------------------------------


C:\Users\sebas\AppData\Local\Temp\ipykernel_33160\3549345714.py:18: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  version = getattr(mod, '__version__', 'instalado')


  Flask                3.1.2            disponible
  Flask-SQLAlchemy     3.1.1            disponible
  SQLAlchemy           2.0.50           disponible
-------------------------------------------------------


C:\Users\sebas\AppData\Local\Temp\ipykernel_33160\3549345714.py:18: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask-SQLAlchemy 3.2. Use feature detection or 'importlib.metadata.version("flask-sqlalchemy")' instead.
  version = getattr(mod, '__version__', 'instalado')


## Introducción a Flask

Flask es un **microframework web** para Python desarrollado por Armin Ronacher y publicado
en 2010 bajo la licencia BSD. Se caracteriza por su diseño minimalista: no impone una
estructura de proyecto específica ni incluye componentes como ORM o autenticación por
defecto, delegando estas decisiones al desarrollador mediante extensiones
(GeeksForGeeks, 2024a).

A diferencia de frameworks de pila completa como Django, Flask expone directamente el ciclo
**WSGI** (*Web Server Gateway Interface*), la interfaz estándar de Python para comunicación
entre servidores web y aplicaciones. Esto facilita su comprensión y lo hace ideal para
aplicaciones pequeñas, microservicios y proyectos académicos (Pallets Projects, 2024).

Las tres extensiones utilizadas en este proyecto son:

| Extensión | Función |
|-----------|---------|
| **Flask-SQLAlchemy** | ORM para gestionar la base de datos SQLite3 |
| **Jinja2** | Motor de plantillas HTML (incluido en Flask) |
| **Gunicorn** | Servidor WSGI de producción (multi-worker) |

## Ejercicio 1: Inicialización de la aplicación Flask

Una aplicación Flask se crea instanciando la clase `Flask` con el argumento `__name__`,
que indica al framework dónde encontrar recursos como plantillas y archivos estáticos.
La configuración de la base de datos se pasa mediante el diccionario `app.config` antes
de inicializar cualquier extensión.

El parámetro `'SQLALCHEMY_DATABASE_URI'` acepta cadenas de conexión en formato estándar.
Para SQLite, la URI `'sqlite:///test.db'` crea el archivo `test.db` dentro de la carpeta
`instance/` del proyecto (comportamiento predeterminado de Flask-SQLAlchemy 3.x).

In [4]:
from flask import Flask

# __name__ indica a Flask la ubicación del modulo raíz del proyecto.
# Flask usa esta referencia para resolver rutas relativas a plantillas y archivos estáticos.
app = Flask(__name__)

# La URI 'sqlite:///test.db' crea el archivo en la carpeta instance/ automáticamente.
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///test.db'

print('Aplicación Flask creada: ' + app.name)
print('URI de base de datos:    ' + app.config['SQLALCHEMY_DATABASE_URI'])
print('Modo debug por defecto:  ' + str(app.debug))

Aplicación Flask creada: __main__
URI de base de datos:    sqlite:///test.db
Modo debug por defecto:  False


## Ejercicio 2: Modelo de datos con Flask-SQLAlchemy

**Flask-SQLAlchemy** es la extensión oficial que integra SQLAlchemy con Flask. Gestiona
automáticamente el ciclo de vida de la sesión de base de datos en cada *request* y expone
`db.Model` como clase base para definir tablas mediante atributos de clase (`db.Column`).

El modelo `TODO` mapea la tabla `todo` de SQLite con cinco columnas:

| Columna | Tipo | Restricción | Descripción |
|---------|------|-------------|-------------|
| `id` | Integer | PRIMARY KEY | Clave primaria autoincrementada |
| `content` | String(200) | NOT NULL | Texto de la tarea |
| `completed` | Boolean | DEFAULT False | Estado de completitud |
| `date_created` | DateTime | AUTO | Fecha de creación |
| `date_modified` | DateTime | AUTO + ONUPDATE | Fecha de última modificación |

`db.func.current_timestamp()` delega la generación de la marca de tiempo al motor de base
de datos, lo que garantiza consistencia independientemente del huso horario del servidor.

In [5]:
from flask import Flask
from flask_sqlalchemy import SQLAlchemy

app = Flask(__name__)
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///test.db'

# db es la instancia central de Flask-SQLAlchemy.
db = SQLAlchemy(app)

class TODO(db.Model):
    id           = db.Column(db.Integer, primary_key=True)
    content      = db.Column(db.String(200), nullable=False)
    completed    = db.Column(db.Boolean, default=False)
    date_created = db.Column(db.DateTime, default=db.func.current_timestamp())
    date_modified= db.Column(db.DateTime, default=db.func.current_timestamp(),
                             onupdate=db.func.current_timestamp())

    def __repr__(self):
        return '<Task %r>' % self.id

# Crear las tablas en la base de datos si no existen
with app.app_context():
    db.create_all()
    columnas = [c.name for c in TODO.__table__.columns]
    print('Tablas creadas exitosamente.')
    print('Columnas de TODO: ' + str(columnas))
    print('Nombre de tabla:  ' + TODO.__tablename__)

Tablas creadas exitosamente.
Columnas de TODO: ['id', 'content', 'completed', 'date_created', 'date_modified']
Nombre de tabla:  todo


## Ejercicio 3: Rutas CRUD con Flask

Flask registra rutas mediante el decorador `@app.route()`. Cada ruta asocia una URL con
una función Python llamada *view function*. Las operaciones CRUD de la aplicación se
distribuyen en cinco rutas:

| Método HTTP | URL | Operación | Descripción |
|-------------|-----|-----------|-------------|
| GET | `/` | Read (listar) | Obtiene todas las tareas ordenadas |
| POST | `/` | Create | Crea una nueva tarea |
| GET | `/complete/<id>` | Update (toggle) | Alterna el estado de completitud |
| GET | `/update/<id>` | Read + Update | Formulario de edición y guardado |
| GET | `/delete/<id>` | Delete | Elimina una tarea por su `id` |

La función `redirect('/')` realiza una redirección HTTP **302**, siguiendo el patrón
**POST/Redirect/GET** para evitar el reenvío accidental de formularios al recargar la página.

`db.session` gestiona las transacciones:
- `db.session.add(obj)` registra un nuevo objeto para inserción.
- `db.session.commit()` persiste todos los cambios pendientes en el archivo `.db`.
- `db.session.delete(obj)` registra un objeto para eliminación.

In [6]:
# Archivo: app.py — Código fuente completo de la aplicación
# Repositorio: https://github.com/RKCbas/todo-flask

from flask import Flask, render_template, request, redirect
from flask_sqlalchemy import SQLAlchemy

# --- Configuración de la aplicación ---
app = Flask(__name__)
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///test.db'
db = SQLAlchemy(app)

# --- Modelo de datos ---
class TODO(db.Model):
    id           = db.Column(db.Integer, primary_key=True)
    content      = db.Column(db.String(200), nullable=False)
    completed    = db.Column(db.Boolean, default=False)
    date_created = db.Column(db.DateTime, default=db.func.current_timestamp())
    date_modified= db.Column(db.DateTime, default=db.func.current_timestamp(),
                             onupdate=db.func.current_timestamp())

    def __repr__(self):
        return '<Task %r>' % self.id

# --- Ruta principal: listar y crear tareas ---
@app.route('/', methods=['POST', 'GET'])
def index():
    if request.method == 'POST':
        task_content = request.form['content']   # Lee el campo 'content' del formulario
        new_task = TODO(content=task_content)
        try:
            db.session.add(new_task)
            db.session.commit()          # Persiste la nueva tarea en el archivo .db
            return redirect('/')         # Patron POST/Redirect/GET
        except Exception:
            return 'There was an issue adding your task'
    else:
        tasks = TODO.query.order_by(TODO.date_created).all()
        return render_template('index.html', tasks=tasks)

# --- Ruta: alternar estado completado ---
@app.route('/complete/<int:id>', methods=['GET'])
def complete(id):
    task = TODO.query.get_or_404(id)     # Devuelve 404 automáticamente si no existe
    task.completed = not task.completed  # Toggle: True -> False, False -> True
    try:
        db.session.commit()
        return redirect('/')
    except Exception:
        return 'There was an issue updating your task'

# --- Ruta: eliminar tarea ---
@app.route('/delete/<int:id>', methods=['GET'])
def delete(id):
    task_to_delete = TODO.query.get_or_404(id)
    try:
        db.session.delete(task_to_delete)
        db.session.commit()
        return redirect('/')
    except Exception:
        return 'There was an issue deleting your task'

# --- Rutas: formulario de edición y guardar cambios ---
@app.route('/update/<int:id>', methods=['GET', 'POST'])
def update(id):
    task = TODO.query.get_or_404(id)
    if request.method == 'POST':
        task.content = request.form['content']  # Sobrescribe el contenido de la tarea
        try:
            db.session.commit()
            return redirect('/')
        except Exception:
            return 'There was an issue updating your task'
    else:
        return render_template('update.html', task=task)

# Crear tablas al iniciar la aplicación
with app.app_context():
    db.create_all()

if __name__ == '__main__':
    app.run(debug=False)

# Mostrar resumen de rutas
with app.app_context():
    print('Rutas registradas:')
    for rule in app.url_map.iter_rules():
        if '/static' not in rule.rule:
            metodos = ','.join(sorted(rule.methods - {'HEAD', 'OPTIONS'}))
            print(f'  [{metodos:<10}] {rule.rule}')

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


Rutas registradas:
  [GET,POST  ] /
  [GET       ] /complete/<int:id>
  [GET       ] /delete/<int:id>
  [GET,POST  ] /update/<int:id>


## Ejercicio 4: Plantillas Jinja2 y herencia de plantillas

Flask utiliza el motor de plantillas **Jinja2** para generar HTML dinámico. Jinja2 permite
insertar variables Python dentro de HTML usando la sintaxis `{{ variable }}`, ejecutar
bloques condicionales con `{% if %}` y ciclos con `{% for %}` (GeeksForGeeks, 2024b).

La **herencia de plantillas** es el mecanismo más importante: `base.html` define la
estructura HTML compartida y declara bloques con `{% block nombre %}`. Las plantillas
hijas usan `{% extends 'base.html' %}` para heredar esa estructura y sobreescriben
únicamente los bloques necesarios, evitando la duplicación de código HTML.

```
templates/
├── base.html      ← Define {% block head %} y {% block body %}
├── index.html     ← Extiende base.html; lista tareas con {% for task in tasks %}
└── update.html    ← Extiende base.html; formulario con valor prellenado
```

Expresiones Jinja2 utilizadas en el proyecto:

| Expresión | Resultado |
|-----------|-----------|
| `{{ task.content }}` | Texto de la tarea |
| `{{ task.date_created.strftime('%b %d') }}` | Fecha formateada (ej: "Jan 01") |
| `{{ tasks\|selectattr('completed')\|list\|length }}` | Cuenta tareas completadas |
| `{% if task.completed %}done{% endif %}` | Clase CSS condicional |
| `{% for task in tasks %}...{% endfor %}` | Iteración sobre la lista de tareas |

In [7]:
import re

# Contenido de las plantillas HTML del proyecto.
# Archivos originales en: templates/base.html, index.html, update.html

base_html = '''<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link rel="icon" type="image/svg+xml" href="{{ url_for(\'static\', filename=\'favicon.svg\') }}">
    <link rel="stylesheet" href="{{ url_for(\'static\', filename=\'css/main.css\') }}">
    {% block head %}{% endblock %}
</head>
<body>
    {% block body %}{% endblock %}
</body>
</html>'''

index_html = '''{% extends 'base.html' %}
{% block head %}<title>Todo List</title>{% endblock %}
{% block body %}
<div class="card">
  <div class="card-header">
    <h1>My Tasks</h1>
    <p>{{ tasks|selectattr('completed')|list|length }} of {{ tasks|length }} completed</p>
  </div>
  <div class="task-list">
    {% if tasks|length < 1 %}
      <div class="empty-state">No tasks yet. Add one below.</div>
    {% else %}
      {% for task in tasks %}
        <div class="task-row {% if task.completed %}done{% endif %}">
          <a href="/complete/{{ task.id }}" class="btn-complete">
            {% if task.completed %}checkmark{% endif %}
          </a>
          <span class="task-content">{{ task.content }}</span>
          <span class="task-date">{{ task.date_created.strftime('%b %d') }}</span>
          <a href="/update/{{ task.id }}" class="btn-edit">edit</a>
          <a href="/delete/{{ task.id }}" class="btn-delete">del</a>
        </div>
      {% endfor %}
    {% endif %}
  </div>
  <form action="/" method="post" class="add-form">
    <input type="text" name="content" placeholder="Add a new task..." required>
    <button type="submit">Add</button>
  </form>
</div>
{% endblock %}'''

update_html = '''{% extends 'base.html' %}
{% block head %}<title>Update Task</title>{% endblock %}
{% block body %}
<div class="card">
  <div class="update-form">
    <a href="/" class="update-back">Back</a>
    <h1>Edit Task</h1>
    <form action="/update/{{ task.id }}" method="post">
      <input type="text" name="content" value="{{ task.content }}" required>
      <button type="submit">Save changes</button>
    </form>
  </div>
</div>
{% endblock %}'''

# Analizar directivas Jinja2 en cada plantilla
print('Plantilla       Variables {{}}   Directivas {%%}')
print('-' * 55)
for nombre, contenido in [('base.html', base_html), ('index.html', index_html), ('update.html', update_html)]:
    variables  = re.findall(r'\{\{.*?\}\}', contenido)
    directivas = re.findall(r'\{%.*?%\}',  contenido)
    print(f'{nombre:<16} {len(variables):<16} {len(directivas)}')
print()
print('Variables encontradas en index.html:')
for v in re.findall(r'\{\{.*?\}\}', index_html):
    print('  ' + v.strip())

Plantilla       Variables {{}}   Directivas {%%}
-------------------------------------------------------
base.html        2                4
index.html       7                14
update.html      2                5

Variables encontradas en index.html:
  {{ tasks|selectattr('completed')|list|length }}
  {{ tasks|length }}
  {{ task.id }}
  {{ task.content }}
  {{ task.date_created.strftime('%b %d') }}
  {{ task.id }}
  {{ task.id }}


## Ejercicio 5: Acceso directo a SQLite3 con el módulo `sqlite3`

Además de Flask-SQLAlchemy, Python incluye en su biblioteca estándar el módulo `sqlite3`
que permite interactuar con bases de datos SQLite mediante SQL puro (GeeksForGeeks, 2024c;
Python Software Foundation, 2024). Esto es útil para inspeccionar la base de datos sin
depender del ORM, o para ejecutar consultas analíticas directas.

El módulo opera con el patrón **Conexión → Cursor → Consulta → Confirmación → Cierre**:

1. `sqlite3.connect('ruta/test.db')` abre o crea el archivo `.db`.
2. `conn.cursor()` crea un cursor para ejecutar sentencias SQL.
3. `cursor.execute(sql)` o `cursor.executemany(sql, datos)` ejecutan la sentencia.
4. `conn.commit()` persiste los cambios (sólo necesario para DML: INSERT, UPDATE, DELETE).
5. `conn.close()` libera el recurso de conexión.

El uso de **parámetros con `?`** (marcadores de posición) en `cursor.execute` previene
inyecciones SQL, ya que SQLite escapa automáticamente los valores.

In [8]:
import sqlite3
from datetime import datetime, timezone

# ':memory:' crea una base de datos temporal en RAM (no afecta el archivo de producción)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# DDL: crear la tabla con el mismo esquema que el modelo ORM
cursor.execute('''
    CREATE TABLE IF NOT EXISTS todo (
        id            INTEGER PRIMARY KEY AUTOINCREMENT,
        content       TEXT    NOT NULL,
        completed     INTEGER NOT NULL DEFAULT 0,
        date_created  TEXT    NOT NULL,
        date_modified TEXT    NOT NULL
    )
''')

# DML: insertar tareas usando parámetros ? (prevención de SQL injection)
ahora = datetime.now(timezone.utc).isoformat()
tareas_ejemplo = [
    ('Configurar entorno Flask',     0, ahora, ahora),
    ('Definir modelo SQLAlchemy',    1, ahora, ahora),
    ('Implementar rutas CRUD',       0, ahora, ahora),
    ('Crear plantillas Jinja2',      1, ahora, ahora),
    ('Desplegar en Render',          0, ahora, ahora),
]
cursor.executemany(
    'INSERT INTO todo (content, completed, date_created, date_modified) VALUES (?, ?, ?, ?)',
    tareas_ejemplo
)
conn.commit()

# DQL: consultar todas las tareas
cursor.execute('SELECT id, content, completed, date_created FROM todo ORDER BY id')
rows = cursor.fetchall()

header = 'ID    Contenido                            Completada   Fecha'
print(header)
print('-' * len(header))
for row in rows:
    estado = 'Si' if row[2] else 'No'
    fecha = str(row[3])[:10]
    line = str(row[0]).ljust(6) + str(row[1]).ljust(37) + estado.ljust(13) + fecha
    print(line)

conn.close()
print('\nConexión cerrada.')

ID    Contenido                            Completada   Fecha
-------------------------------------------------------------
1     Configurar entorno Flask             No           2026-06-05
2     Definir modelo SQLAlchemy            Si           2026-06-05
3     Implementar rutas CRUD               No           2026-06-05
4     Crear plantillas Jinja2              Si           2026-06-05
5     Desplegar en Render                  No           2026-06-05

Conexión cerrada.


## Ejercicio 6: Consultas avanzadas e introspección de esquema en SQLite3

SQLite expone metadatos del esquema a través del pragma `PRAGMA table_info(nombre_tabla)`,
equivalente al comando `DESCRIBE` de MySQL. También permite consultas agregadas estándar
como `COUNT()`, `SUM()` y `AVG()`.

La tabla interna `sqlite_master` lista todos los objetos de la base de datos (tablas,
índices, vistas), lo que es útil para validar que el esquema generado por el ORM coincide
con el esperado.

In [9]:
import sqlite3
from datetime import datetime, timedelta
import random

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE todo (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        content TEXT NOT NULL,
        completed INTEGER NOT NULL DEFAULT 0,
        date_created TEXT NOT NULL,
        date_modified TEXT NOT NULL
    )
''')

base_dt = datetime(2025, 1, 1)
random.seed(42)
datos = [
    ('Tarea ' + str(i+1), random.randint(0, 1),
     (base_dt + timedelta(days=i)).isoformat(),
     (base_dt + timedelta(days=i)).isoformat())
    for i in range(10)
]
cursor.executemany(
    'INSERT INTO todo (content, completed, date_created, date_modified) VALUES (?, ?, ?, ?)',
    datos
)
conn.commit()

# Inspección de esquema con PRAGMA
print('=== Esquema de la tabla todo (PRAGMA table_info) ===')
cursor.execute('PRAGMA table_info(todo)')
cols = cursor.fetchall()
print('cid  nombre          tipo       notnull  default      pk')
print('-' * 55)
for col in cols:
    print(str(col[0]).ljust(5) + str(col[1]).ljust(16) + str(col[2]).ljust(11) +
          str(col[3]).ljust(9) + str(col[4]).ljust(13) + str(col[5]))

# Consultas agregadas
print('\n=== Estadisticas de tareas ===')
cursor.execute('SELECT COUNT(*) FROM todo')
total = cursor.fetchone()[0]
cursor.execute('SELECT COUNT(*) FROM todo WHERE completed = 1')
completadas = cursor.fetchone()[0]
pendientes = total - completadas
print('Total de tareas:       ' + str(total))
print('Tareas completadas:    ' + str(completadas))
print('Tareas pendientes:     ' + str(pendientes))
print('Porcentaje completado: ' + str(round(completadas/total*100, 1)) + '%')

# Listar tablas en sqlite_master
print('\n=== Objetos en sqlite_master ===')
cursor.execute("SELECT type, name FROM sqlite_master WHERE type='table'")
for obj in cursor.fetchall():
    print('  ' + str(obj[0]) + ': ' + str(obj[1]))

conn.close()

=== Esquema de la tabla todo (PRAGMA table_info) ===
cid  nombre          tipo       notnull  default      pk
-------------------------------------------------------
0    id              INTEGER    0        None         1
1    content         TEXT       1        None         0
2    completed       INTEGER    1        0            0
3    date_created    TEXT       1        None         0
4    date_modified   TEXT       1        None         0

=== Estadisticas de tareas ===
Total de tareas:       10
Tareas completadas:    2
Tareas pendientes:     8
Porcentaje completado: 20.0%

=== Objetos en sqlite_master ===
  table: todo
  table: sqlite_sequence


## Ejercicio 7: Despliegue en producción con Gunicorn y Render

El servidor de desarrollo integrado de Flask (`app.run(debug=True)`) no es apto para
producción: es monohilo, no está optimizado para concurrencia y expone información de
depuración. **Gunicorn** (*Green Unicorn*) es un servidor WSGI pre-fork de múltiples
trabajadores que expone la aplicación Flask como un servidor HTTP de producción.

El archivo **Procfile** es la convención usada por plataformas PaaS como Render o Heroku
para declarar los comandos de arranque de cada proceso:

```
web: gunicorn app:app
```

Donde `app:app` significa: *del módulo `app.py`, usa el objeto llamado `app`*.

**Render** detecta automáticamente el `Procfile`, instala las dependencias desde
`requirements.txt` y expone el puerto asignado. La aplicación en producción está
disponible en: https://todo-flask-zkex.onrender.com

| Aspecto | Desarrollo | Producción |
|---------|------------|------------|
| Servidor | `flask run` (Werkzeug) | `gunicorn app:app` |
| Workers | 1 (monohilo) | Múltiples (configurable) |
| Debug | Habilitado | Deshabilitado |
| Recarga automática | Sí | No |

In [10]:
import re

requirements_txt = '''Flask==3.1.2
Flask-SQLAlchemy==3.1.1
gunicorn==23.0.0
'''

procfile = 'web: gunicorn app:app'

print('=== requirements.txt ===')
print(requirements_txt)
print('=== Procfile ===')
print(procfile)
print()

print('=== Analisis de dependencias ===')
paquetes = re.findall(r'^(\S+)==(\S+)', requirements_txt, re.MULTILINE)
roles = {
    'Flask': 'Framework web principal',
    'Flask-SQLAlchemy': 'Integración ORM con Flask',
    'gunicorn': 'Servidor WSGI de producción',
}
print('Paquete                Version      Rol')
print('-' * 60)
for nombre, version in paquetes:
    rol = roles.get(nombre, 'Dependencia auxiliar')
    print(nombre.ljust(23) + ('v' + version).ljust(13) + rol)

=== requirements.txt ===
Flask==3.1.2
Flask-SQLAlchemy==3.1.1
gunicorn==23.0.0

=== Procfile ===
web: gunicorn app:app

=== Analisis de dependencias ===
Paquete                Version      Rol
------------------------------------------------------------
Flask                  v3.1.2       Framework web principal
Flask-SQLAlchemy       v3.1.1       Integración ORM con Flask
gunicorn               v23.0.0      Servidor WSGI de producción


## Descripción

La aplicación implementada integra los tres niveles clásicos de una arquitectura web: la
capa de presentación (plantillas Jinja2), la capa de lógica de negocio (*view functions*
de Flask) y la capa de persistencia (modelo ORM sobre SQLite3). El uso de Flask-SQLAlchemy
permite abstraer completamente el SQL en el código Python mediante el patrón Active Record:
`db.session.add()`, `db.session.commit()` y `db.session.delete()` se traducen
automáticamente a sentencias `INSERT`, `UPDATE` y `DELETE`, mientras que
`TODO.query.order_by(TODO.date_created).all()` genera un `SELECT ... ORDER BY`.
Esta abstracción reduce el riesgo de errores de sintaxis SQL y hace el código más portable
entre motores de base de datos.

El patrón **POST/Redirect/GET** implementado en todas las rutas que reciben formularios es
una práctica estándar del desarrollo web: después de procesar un `POST`, la aplicación
responde con `redirect('/')` hacia la página de listado en lugar de renderizar directamente
la respuesta. Esto evita que el navegador reenvíe el formulario al recargar la página,
previniendo la creación accidental de tareas duplicadas. La herencia de plantillas Jinja2
complementa este diseño al centralizar el HTML estructural en `base.html`, permitiendo que
`index.html` y `update.html` sólo definan el contenido específico de cada vista mediante
bloques `{% block body %}`.

El despliegue en Render a través de Gunicorn demuestra la separación entre los entornos de
desarrollo y producción. El archivo `Procfile` declara el comando `gunicorn app:app`, que
lanza múltiples workers WSGI capaces de manejar solicitudes concurrentes, a diferencia del
servidor monohilo de desarrollo de Flask. La URI `sqlite:///test.db` crea el archivo
dentro de la carpeta `instance/` del proyecto (comportamiento de Flask-SQLAlchemy 3.x),
siguiendo las recomendaciones de seguridad de la documentación oficial para separar los
datos de instancia del código fuente.

## Referencias

### Flask y Desarrollo Web con Python

1. Pallets Projects. (2024). *Flask Documentation (3.x)*. Documentación oficial de Flask.
   https://flask.palletsprojects.com/en/stable/

2. GeeksForGeeks. (2024a). *Flask Tutorial*. GeeksForGeeks.
   https://www.geeksforgeeks.org/flask-tutorial/

3. GeeksForGeeks. (2024b). *Flask – Rendering Templates*. GeeksForGeeks.
   https://www.geeksforgeeks.org/flask-rendering-templates/

### SQLite3 y Persistencia de Datos

4. GeeksForGeeks. (2024c). *Python SQLite*. GeeksForGeeks.
   https://www.geeksforgeeks.org/python-sqlite/

5. Python Software Foundation. (2024). *sqlite3 — DB-API 2.0 interface for SQLite databases*.
   Documentación oficial de Python 3 (es).
   https://docs.python.org/es/3/library/sqlite3.html

### Repositorios del Proyecto

6. RKCbas. (2024). *todo-flask — Aplicación web de lista de tareas con Flask y SQLite3*
   [Repositorio de GitHub].
   https://github.com/RKCbas/todo-flask

7. RKCbas. (2024). *Maestría en Inteligencia Artificial — Prácticas* [Repositorio de GitHub].
   https://github.com/RKCbas/Maestria-en-Inteligencia-Artificial---Practicas

### Aplicación Desplegada

8. RKCbas. (2024). *Todo Flask — Aplicación desplegada en Render*.
   https://todo-flask-zkex.onrender.com